# Carrier Allocation Data Consolidation

Scans Excel files in configured lakehouse folders, identifies allocation worksheets
matching the `{Carrier} Alloc YYYY.MM.DD` pattern, extracts the data rows
(skipping metadata headers and summary footers), and consolidates everything
into a single delta table.

**Key behaviors:**
- `_02. TO BE TRANSFERRED`: processes only root-level `.xlsx` files (ignores year subfolders)
- `Mills TO BE TRANSFERRED`: processes all `.xlsx` files recursively
- Files with carrier code `UHC` in their name are skipped
- Header row is detected dynamically by scanning for `MEMBER NAME` in column A
- Summary/footer rows below the data are excluded automatically
- All extracted columns are stored as nullable strings in the delta table for
  schema-stable consolidation across heterogeneous carrier files; cast to
  date / decimal in downstream queries as needed.

In [ ]:
# ============================================================
# CONFIGURATION - adjust these values for your environment
# ============================================================

# Local mount path to the Lakehouse Files section.
LAKEHOUSE_FILES_PATH = "/lakehouse/default/Files"

# Folders to scan for Excel files.
# Set "root_only" to True to skip subfolders (e.g., year folders).
SCAN_FOLDERS = [
    {"path": "_02. TO BE TRANSFERRED", "root_only": True},
    {"path": "Mills TO BE TRANSFERRED", "root_only": False},
]

# Delta table name - written to the default lakehouse Tables section.
# Change this to your desired table name.
DELTA_TABLE_NAME = "consolidated_carrier_alloc"

In [ ]:
import os
import re
import logging
import warnings
from datetime import datetime, date

from openpyxl import load_workbook
import pandas as pd
from pyspark.sql.functions import current_timestamp

# openpyxl prints a UserWarning for the "Data Validation extension" on
# every workbook open; it is harmless and just clutters the log.
warnings.filterwarnings(
    "ignore",
    message="Data Validation extension is not supported and will be removed",
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("carrier_alloc")

In [ ]:
# ============================================================
# CONSTANTS
# ============================================================

# Regex to match allocation worksheet names: "{Carrier} Alloc YYYY.MM.DD"
ALLOC_SHEET_RE = re.compile(r"^(.+?)\s+Alloc\s+(\d{4}\.\d{2}\.\d{2})$")

# Carrier codes to skip - any file whose name contains one of these is excluded.
SKIP_CARRIER_CODES = {"UHC"}

# Header marker - the value expected in column A of the header row.
HEADER_MARKER = "MEMBER NAME"

# Maximum rows to scan from the top when looking for the header.
MAX_HEADER_SCAN = 50

# Valid carrier codes (for reference / optional validation).
VALID_CARRIER_CODES = {
    "BLUE IL", "BLUE MI", "BLUE MT", "BLUE TN", "BLUE TX", "BLUE",
    "BLUE RM", "BLUE VB", "BND SP", "BND VB", "CENT", "CENT VB",
    "CIGN HS", "CIGN MS", "CLEA", "CLOV RIM", "COMB", "CONT",
    "CORE RM", "DEVO", "DPBR", "EMBL", "EXCE RIM", "FLOR", "GUAR",
    "GEOB", "GLIC", "GLOB", "HFIR", "HFIR RIM", "HEAL", "HIGH RIM",
    "HORI", "HUMA CV", "HUMA NSG", "HUMA RM", "HUMA TAIA",
    "HUMA TAIA OR", "IMPE RIM", "MEDI", "MOLI", "MWG", "MOO",
    "MOO PDP", "O&A", "PHYS", "REGE RIM", "REGE", "RITTIM", "SCAN",
    "SILV RIM", "SILV SMS", "SPAR", "TRAN", "TRUE", "TRUS", "UA",
    "UHC", "UHC CV", "UHC RM", "VB", "VNS RIM", "VSP", "WHA",
    "WELL CE", "WELL NSG", "WELL SP", "WELL CV", "WELL RM",
}

In [ ]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================


def _is_valid_xlsx(filename):
    """Return True for real .xlsx files, False for temp/hidden files."""
    return filename.lower().endswith(".xlsx") and not filename.startswith("~$")


def discover_excel_files(base_path, folder_cfg):
    """Return a sorted list of .xlsx file paths in a lakehouse folder."""
    folder = os.path.join(base_path, folder_cfg["path"])
    if not os.path.isdir(folder):
        log.warning("Folder not found: %s", folder)
        return []

    files = []

    if folder_cfg.get("root_only", False):
        for name in os.listdir(folder):
            full = os.path.join(folder, name)
            if os.path.isfile(full) and _is_valid_xlsx(name):
                files.append(full)
    else:
        for dirpath, _, filenames in os.walk(folder):
            for name in filenames:
                if _is_valid_xlsx(name):
                    files.append(os.path.join(dirpath, name))

    return sorted(files)


def should_skip_file(filepath):
    """Return True if the filename contains a carrier code we want to skip."""
    name_upper = os.path.basename(filepath).upper()
    return any(code in name_upper for code in SKIP_CARRIER_CODES)


def find_alloc_sheets(sheetnames):
    """Return (sheet_name, carrier_code, alloc_date) for matching sheets.

    Takes a list of sheet names rather than a workbook object so it can
    be called without keeping the workbook open.
    """
    results = []
    for name in sheetnames:
        m = ALLOC_SHEET_RE.match(name)
        if m:
            results.append((name, m.group(1).strip(), m.group(2)))
    return results


def extract_sheet_data(ws):
    """Detect the header row and extract all data rows from a worksheet.

    Strategy:
        1. Scan rows from the top for "MEMBER NAME" in column A -> header row.
        2. Build unique column names from the header (duplicates get a suffix,
           unnamed columns get a "_col_N" placeholder).
        3. Collect every subsequent row whose column A is non-empty. Empty
           column-A rows are either blank spacers or summary/footer rows,
           so they are safely skipped.

    Returns:
        (header_row_excel, col_names, records)
            header_row_excel: 1-based Excel row number of the header,
                              or None if not found.
            col_names:        list of unique column names.
            records:          list of dicts (one per data row).
    """
    all_rows = list(ws.iter_rows(values_only=True))

    # --- Locate header row ---
    header_idx = None
    for i, row in enumerate(all_rows[:MAX_HEADER_SCAN]):
        if row and row[0] is not None:
            if str(row[0]).strip().upper() == HEADER_MARKER:
                header_idx = i
                break

    if header_idx is None:
        return None, [], []

    header = list(all_rows[header_idx])
    n_cols = len(header)

    # --- Build unique column names (case-insensitive dedup) ---
    # Spark/Delta is case-insensitive by default and enforces it for column
    # names (Parquet is case-sensitive but Delta is case-preserving), so two
    # headers like "CARRIER" and "Carrier" would collide on saveAsTable with
    # COLUMN_ALREADY_EXISTS. Bucket by the lowercase form so the second
    # occurrence picks up a "_2" suffix; original casing is preserved on the
    # first occurrence so existing analytics keep working.
    col_names = []
    seen = {}  # key: lowercase name, value: occurrence count
    for j in range(n_cols):
        raw = header[j]
        base = (
            str(raw).strip()
            if raw is not None and str(raw).strip()
            else f"_col_{j + 1}"
        )
        key = base.lower()
        if key not in seen:
            seen[key] = 1
            col_names.append(base)
        else:
            seen[key] += 1
            col_names.append(f"{base}_{seen[key]}")

    # --- Collect data rows ---
    records = []
    for row in all_rows[header_idx + 1:]:
        if not row or row[0] is None or str(row[0]).strip() == "":
            continue  # skip blank / summary rows

        padded = list(row)[:n_cols]
        padded += [None] * (n_cols - len(padded))
        records.append(dict(zip(col_names, padded)))

    return header_idx + 1, col_names, records  # 1-based Excel row

In [ ]:
# ============================================================
# MAIN PROCESSING
# ============================================================

all_records = []
stats = {
    "files_found": 0,
    "files_skipped_uhc": 0,
    "sheets_matched": 0,
    "rows_extracted": 0,
    "errors": 0,
}

for cfg in SCAN_FOLDERS:
    files = discover_excel_files(LAKEHOUSE_FILES_PATH, cfg)
    log.info("Folder '%s' -- %d Excel file(s) found", cfg["path"], len(files))
    stats["files_found"] += len(files)

    for fpath in files:
        fname = os.path.basename(fpath)

        if should_skip_file(fpath):
            log.info("  SKIP  %s  (UHC carrier)", fname)
            stats["files_skipped_uhc"] += 1
            continue

        try:
            wb = load_workbook(fpath, read_only=True, data_only=True)
        except Exception as exc:
            log.error("  ERROR opening %s: %s", fname, exc)
            stats["errors"] += 1
            continue

        try:
            alloc_sheets = find_alloc_sheets(wb.sheetnames)
            if not alloc_sheets:
                log.debug("  No alloc sheets in %s", fname)
                continue

            for sheet_name, carrier_code, alloc_date in alloc_sheets:
                ws = wb[sheet_name]
                _, _, records = extract_sheet_data(ws)

                if not records:
                    log.warning(
                        "  No data rows in '%s' (%s)", sheet_name, fname
                    )
                    continue

                # Attach metadata columns to each record.
                alloc_date_iso = alloc_date.replace(".", "-")
                for rec in records:
                    rec["_source_file"] = fname
                    rec["_sheet_name"] = sheet_name
                    rec["_carrier_code"] = carrier_code
                    rec["_alloc_date"] = alloc_date_iso

                all_records.extend(records)
                stats["sheets_matched"] += 1
                stats["rows_extracted"] += len(records)
                log.info(
                    "  >> '%s' -- %d rows extracted", sheet_name, len(records)
                )
        finally:
            wb.close()

log.info("Processing complete: %s", stats)

In [ ]:
# ============================================================
# SAVE TO DELTA TABLE
# ============================================================

if not all_records:
    log.warning("No records extracted. Delta table will not be created.")
else:
    pdf = pd.DataFrame(all_records)

    # Move metadata columns to the end for readability.
    meta_cols = ["_source_file", "_sheet_name", "_carrier_code", "_alloc_date"]
    data_cols = [c for c in pdf.columns if c not in meta_cols]
    pdf = pdf[data_cols + meta_cols]

    # ----- Normalize types for reliable Spark conversion -----
    # Excel cells in the same column position are inconsistently typed
    # across rows and across carrier files: a "PREMIUM PAID" cell may be
    # a float in one row, "" in another, and None in a third. The Arrow
    # converter that backs spark.createDataFrame raises ArrowInvalid on
    # such mixed object columns. Coercing every object-dtype column to a
    # nullable string is the safest universal cast - it preserves the value,
    # allows nulls, produces a stable schema across all carrier files, and
    # avoids NullType columns when a column happens to be entirely empty.
    # Cast columns to date / decimal in downstream queries as needed.
    def _to_str_or_null(v):
        if v is None:
            return None
        if isinstance(v, float) and pd.isna(v):
            return None
        return str(v)

    for col in pdf.columns:
        if pdf[col].dtype == object:
            pdf[col] = pdf[col].map(_to_str_or_null).astype("string")

    # ----- Convert to Spark and write -----
    sdf = spark.createDataFrame(pdf)
    sdf = sdf.withColumn("_ingested_at", current_timestamp())

    (
        sdf.write
        .mode("overwrite")
        .format("delta")
        .option("overwriteSchema", "true")
        .saveAsTable(DELTA_TABLE_NAME)
    )

    log.info(
        "Saved %d rows x %d columns to delta table '%s'",
        sdf.count(),
        len(sdf.columns),
        DELTA_TABLE_NAME,
    )

In [ ]:
# ============================================================
# VERIFY RESULTS
# ============================================================

df = spark.read.table(DELTA_TABLE_NAME)
print(f"Table:   {DELTA_TABLE_NAME}")
print(f"Rows:    {df.count()}")
print(f"Columns: {len(df.columns)}")
print()
display(df.limit(20))